# 从设备接线到 GreenMOT＋Ramsey

这是一条短的入门路径：**设备库 → 接线与信号 → Qubit → 原生准备/读出 → Ramsey → 编译和波形**。
下面的数据库、descriptor、Qubit 和原生入口都先完整展示，允许直接修改。最后才介绍如何分文件保存。

默认 Run All **只离线编译**，不连接仪器。GreenMOT 使用现有 TTL kernel 演示包；它不制备原子。
频率、幅度、脉宽和触发延迟均为示例，不代表物理校准。读出函数触发相机，不自动取得图像。

环境：项目根目录执行 `uv sync --locked --group notebooks`，然后 `uv run --locked --group notebooks jupyter lab`。
选择项目 CPython 3.12 kernel。只有真实 ARTIQ 编译/运行才需要独立的官方工具链。
本教程不需要先学习 RB、磁盘编译、bank、运行数据库或 Spectrum SDK。

In [ ]:
import json
import pprint
from pathlib import Path
from types import SimpleNamespace

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from taiyidsl import (
    DeviceSetup,
    Experiment,
    LabConfig,
    NativeKernelPackage,
    Qubit,
    QubitOperations,
    SweepParameter,
    deg,
    delay,
    kHz,
    ms,
    ns,
    qubit_operation,
    ramsey_experiment,
    reset_phase,
    section,
    us,
)

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/taiyidsl").is_dir()
)
OUTPUT = ROOT / "output/simple_ramsey"
OUTPUT.mkdir(parents=True, exist_ok=True)


def table(rows):
    if not rows:
        return
    keys = list(rows[0])

    def cell(value):
        return str(value).replace("|", r"\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(keys) + " |",
        "| " + " | ".join(["---"] * len(keys)) + " |",
    ]
    lines += [
        "| " + " | ".join(cell(row.get(key, "")) for key in keys) + " |" for row in rows
    ]
    display(Markdown("\n".join(lines)))

## 1. 两个设备库：设备是什么，在哪里

`device_db` 沿用 ARTIQ 的格式。`core.arguments.host` 是 core 地址，TTL 的 `channel` 必须与实际 gateware 一致。
此处 `host=None` 明确表示离线配置。两个 AWG trigger 与 MOT/camera 的 TTL 是不同设备。

下面的字典就是 `device_db.py` 的完整示例内容；不会 import ARTIQ 驱动。

In [ ]:
"""OFFLINE example: replace addresses and channels with the lab's device_db."""

device_db = {
    "core": {
        "type": "local",
        "module": "artiq.coredevice.core",
        "class": "Core",
        "arguments": {"host": None, "ref_period": 1e-9, "target": "rv32g"},
    },
    "awg_trigger_a": {
        "type": "local",
        "module": "artiq.coredevice.ttl",
        "class": "TTLOut",
        "arguments": {"channel": 0},
    },
    "awg_trigger_b": {
        "type": "local",
        "module": "artiq.coredevice.ttl",
        "class": "TTLOut",
        "arguments": {"channel": 1},
    },
    "ttl_556_3DMOT_Sh": {
        "type": "local",
        "module": "artiq.coredevice.ttl",
        "class": "TTLOut",
        "arguments": {"channel": 2},
    },
    "ttl_Camera_Trig": {
        "type": "local",
        "module": "artiq.coredevice.ttl",
        "class": "TTLOut",
        "arguments": {"channel": 3},
    },
}

`spec_db` 是 TaiyiDSL 的卡配置格式，内部通过 Spectrum 官方 `spcm` 驱动执行。
本例 A 卡启用一个输出，B 卡启用两个输出。UID `A/B` 用于接线；`serial_number` 才用于核对真实卡身份。
实际环境可在对应卡条目增加 `address`，例如官方驱动接受的 `/dev/spcm0` 标识；同时仍核对序列号。
`output_range_mv` 是模拟输出量程，不能把它与门的归一化 `pi_amplitude` 混为一谈。

In [ ]:
"""Project configuration, not a Spectrum SDK database; no hardware is opened."""

spec_db = {
    "A": {
        "serial_number": "SIM_A",
        "model": "M4i.6631-x8",
        "active_channels": [0],
        "sample_rate_hz": 1_250_000_000,
        "reference_clock_hz": 125_000_000,
        "output_range_mv": 500,
        "trigger_latency_samples": 500,  # illustrative, NOT measured
    },
    "B": {
        "serial_number": "SIM_B",
        "model": "M4i.6631-x8",
        "active_channels": [0, 1],
        "sample_rate_hz": 1_250_000_000,
        "reference_clock_hz": 125_000_000,
        "output_range_mv": 500,
        "trigger_latency_samples": 520,  # illustrative, NOT measured
    },
}

## 2. 完整 descriptor：谁连接谁

这是可直接编辑的接线定义，没有隐藏在辅助函数中：

- `instruments` 引用上述数据库，不再次填写地址和时钟。
- `channels` 是实验室命名的物理线路；`nuclear_rf` 在 A 卡输出 0，clock/rydberg 在 B 卡。
- `sync_groups` 定义 A/B 分别接哪一个 ARTIQ TTL。
- `signals` 将 Qubit 逻辑信号连到实验室线路。`q0.rf` 是逻辑名，不是卡槽编号。

本 Ramsey 只用 RF，B 卡仍属于同一同步组，会播放对应的零输出程序。后续在该组使用 clock/rydberg 时，它们共享同一时间轴。

In [ ]:
"""Signal roles and physical wiring, separate from device addresses/calibration."""

descriptor = {
    "name": "ramsey_teaching_lab",
    "backend": "artiq-spectrum",
    "instruments": {
        "core": {"kind": "artiq", "device": "core"},
        "A": {"kind": "spectrum", "device": "A"},
        "B": {"kind": "spectrum", "device": "B"},
    },
    "channels": {
        "nuclear_rf": {"instrument": "A", "channel": 0},
        "clock": {"instrument": "B", "channel": 0},
        "rydberg": {"instrument": "B", "channel": 1},
        "camera": {"instrument": "core", "device": "ttl_Camera_Trig"},
    },
    "sync_groups": {
        "qubit_awg": {
            "cards": ["A", "B"],
            "triggers": {"A": "awg_trigger_a", "B": "awg_trigger_b"},
        },
    },
    "signals": {
        "q0.rf": "nuclear_rf",
        "q0.clock": "clock",
        "q0.rydberg": "rydberg",
        "camera": "camera",
    },
}

In [ ]:
setup = DeviceSetup.from_descriptor(descriptor, device_db=device_db, spec_db=spec_db)
table(setup.connections())
table(setup.trigger_connections())

配置稳定后，可以把两个数据库分别保存为 `.py` 文件，再传入路径。加载器静态读取赋值和数字表达式，不执行配置代码；动态生成的设备库可显式传入已求值字典。
这一步只读取配置，没有打开驱动或连接仪器。`SignalMap` 和 Spectrum 分组由 setup 内部生成。

In [ ]:
config_dir = OUTPUT / "lab_config"
config_dir.mkdir(exist_ok=True)
for name, value in (("device_db", device_db), ("spec_db", spec_db)):
    (config_dir / f"{name}.py").write_text(
        name + " = " + pprint.pformat(value, sort_dicts=False) + "\n", encoding="utf-8"
    )
setup = DeviceSetup.from_descriptor(
    descriptor,
    device_db=config_dir / "device_db.py",
    spec_db=config_dir / "spec_db.py",
)

## 3. Qubit：信号与校准参数

`q0.rf` 是信号，`q0.parameters.rf` 是校准参数。每个驱动采用相同层级；不再把 RF 放一处、clock 放另一处。
`frequency` 是波形驱动频率，`resonance_frequency` 描述物理跃迁。对于 AOM/混频链路，两者不一定相同，不能自动替代。
π 和 π/2 的幅度、时间独立给出，不根据半幅度或半时长推测已校准。
clock/rydberg 此处为空，表示尚未提供门参数；调用相关门会明确报缺参。

In [ ]:
q0 = Qubit(
    uid="q0",
    signals={"rf": "q0.rf", "clock": "q0.clock", "rydberg": "q0.rydberg"},
    parameters={
        "rf": {
            "frequency": 30 * kHz,
            "resonance_frequency": 30 * kHz,
            "pi_amplitude": 0.2,
            "pi2_amplitude": 0.2,
            "pi_duration": 200 * us,
            "pi2_duration": 100 * us,
            "waveform": "square",
            "frame": "spin",
        },
        "clock": {},  # no invented clock calibration; not used by this Ramsey
        "rydberg": {},
        "extensions": {},
    },
    position=None,
)

In [ ]:
print("RF 信号:", q0.rf)
print("RF π/2 幅度:", q0.parameters.rf.pi2_amplitude)
print("RF π/2 时间 [s]:", q0.parameters.rf.pi2_duration)
print("clock 信号:", q0.clock)
lab = LabConfig(setup=setup, qubits=[q0], emulation=True)

Qubit 是不可变快照。单次门覆盖不会改写校准；正式更新使用 `QubitStore`，保留 revision 与来源。
位置可设为 `{"x": 0, "y": 0, "z": 0, "coordinate_system": "lab"}`，长度单位为米；它不会自动移动原子或建立寻址。

若将 `q1.rf` 也连接 `nuclear_rf`，必须把该 channel 标为 `sharing="global"`。这是公共场，不能独立驱动 q0/q1。
此时使用 `ops.global_x90(q0, q1, calibration=q0)`：只发一次波形，以 q0 的参数与 frame 为准，记录 q1 也受影响，不保证 q1 同样旋转 90°。

## 4. 原生 ARTIQ 包：登记一次，正文调用

这里复用已有 `GreenMOTFixture` 类。库读取它的源码和静态依赖，不在 notebook 中 import 或执行 ARTIQ kernel。
`.kernel(...)` 只是创建一个“将来在生成 kernel 中调用此方法”的 DSL 代理。

准备过程分为可变前段和有明确结束边界的固定尾段；两段合成 `prepare_MOT()`。
固定尾段的返回游标会检查，不能通过暗加 delay 掩盖不符。`finalize_method` 是旧接口的每 shot 清理，不是会话结束清理。

In [ ]:
package = NativeKernelPackage.from_package(
    root=ROOT / "examples/artiq_spectrum/greenmot_native",
    entry="greenmot_fixture:GreenMOTFixture",
    resources=("ttl_556_3DMOT_Sh", "ttl_Camera_Trig"),
    initialize_method="initialize_native",
    finalize_method="finalize_native",
)
prepare_prefix = package.kernel("prepare_MOT", timing="prefix")
prepare_tail = package.kernel(
    "finish_preparation",
    timing="timed",
    duration=15 * ms + 16 * ns,
    end_boundary="Fixture TTL falling edge; no analog preparation model",
)
readout = package.kernel("execute_readout", timing="suffix", duration=100.48 * ms)


def prepare_MOT():
    prepare_prefix()
    prepare_tail()


native = SimpleNamespace(prepare_MOT=prepare_MOT, readout=readout)

接真实 GreenMOT 时，使用现有 `GreenMOTAdapter(greenmot_source=实际源文件路径)` 替换 `native`。
适配器保留原准备前段、15 ms＋16 ns 固定尾段及每 shot 清理，并检查拆分未改变原操作顺序；不要把未知时长的整个原程序随意标为 fixed duration。
真实尾段结束点是 Zotino 的数字 LDAC 边沿，不等于模拟磁场已经稳定；`preparation_to_gate` 需要实验室确定。

其他原生类按同样方式登记 `root`、`entry`、资源和方法。若方法包含 reset/break/wait，不能直接声明成严格 timed/suffix。
原生方法放在 realtime 的准备或读出位置，不放进 AWG gate Section。基础 Ramsey 不需要 NativeProgram 或 compile_stream。

## 5. 操作与 Section：只用一种可复用容器

`ops.x90(q0)` 从校准参数生成一个 π/2 操作，其返回值是 Section。装饰器函数体调用普通 DSL，返回 None；装饰器收集节点并返回 Section。
在活动上下文中调用会自动插入一次；下面在上下文外定义模板，只构建，不执行。

In [ ]:
ops = QubitOperations(drive="rf")


@qubit_operation
def two_pulses(q, tau):
    reset_phase(q.phase_frame("spin"))
    ops.x90(q)
    delay(q.rf, tau)
    ops.x90(q, phase=90 * deg)


template = two_pulses(q0, 20 * us)
print("可复用容器:", type(template).__name__)
print("第一个门:", type(ops.x90(q0)).__name__)

默认串行。需要多个独立输出对齐时，用 `section(..., schedule="parallel", align="start"/"end", qubits=(...))`。
直接子操作是完整并行分支，复合分支在内部串行；同一公共输出不能作为两条独立分支。
`layer()` 只保留旧接口包装。`barrier()` 是 ASAP 中的同步点，没有子内容，也不负责右对齐；本例不需要它。

例如两个独立 qubit 的门结束对齐：
```python
with section("finish_together", schedule="parallel", align="end", qubits=(q0, q1)):
    ops.x90(q0)
    ops.y90(q1)
```
Section 与 segment 都不自动清 frame；Ramsey 的 reset_phase 在门体入口明确写出。

## 6. 完整 Ramsey 实验正文

先看完整正文，不隐藏准备、扫描和 gate sequence 边界。`tau` 是第一脉冲结束到第二脉冲开始的时间。
两个脉冲都使用校准频率加 detuning；第二脉冲只额外加 second_phase。载波沿实际时间轴继续演化，不能再重复加一遍 `2π·detuning·tau`。

In [ ]:
delays = SweepParameter("tau", [1, 20, 80], "us")
detuning = 1 * kHz
second_phase = 90 * deg
drive_frequency = q0.parameters.rf.frequency + float(detuning.value)

exp = Experiment("ramsey_explicit", [q0])
with exp.average(2):
    with exp.sweep(delays):
        with exp.realtime():
            native.prepare_MOT()
            delay(10 * us)
            with exp.triggered_sequence(group="qubit_awg"):
                with section("ramsey"):
                    reset_phase(q0.phase_frame("spin"))
                    ops.x90(q0, frequency=drive_frequency, phase=0 * deg)
                    delay(q0.rf, delays)
                    ops.x90(q0, frequency=drive_frequency, phase=second_phase)
            native.readout()

日常复用可改用标准构造函数，detuning 和 second_phase 就是实验参数，不需要修改全局校准。
也支持把它们设为 SweepParameter；同时扫描时顺序为 delay → detuning → second_phase。
下面采用这一写法进行一次离线编译，接下来的查看都复用这份产物。

In [ ]:
exp = ramsey_experiment(
    qubit=q0,
    native=native,
    delays=delays,
    detuning=detuning,
    second_phase=second_phase,
    shots=2,
    preparation_to_gate=10 * us,
    group="qubit_awg",
)
compiled = lab.compile(exp)
print("扫描点:", len(compiled.points), "；shots/point:", compiled.shots)
table(compiled.axes)
assert q0.parameters.rf.frequency == 30_000

## 7. 检查实际编译结果：时间、相位和最终 DAC

下面的报告源于编译产物，波形查看器解码最终 packed DAC 样本。
它不是把理想门图当成硬件输出，也不是测量数据。
注意：ARTIQ 原生 kernel 的波形没有自动模拟；报告保留其调用、时长和源码。

In [ ]:
report = compiled.export_report(OUTPUT / "notebook_report")
display(Markdown(f"[打开本次实验报告]({report['index'].resolve().as_uri()})"))
report_data = json.loads(
    (OUTPUT / "notebook_report/report.json").read_text(encoding="utf-8")
)
print("报告路径:", report["index"].resolve())

### 32 点约束属于 segment，不属于每一个门

当前 1.25 GS/s 的采样间隔是 0.8 ns。segment 每通道长度必须是 32 的整数倍；启用一个通道时最少 384 点，两个通道时最少 192 点。
编译器先建立完整门时间轴，再切分 segment。不能为了凑 32 点在两个 Ramsey 脉冲之间加零或拉长 tau。
末尾保留零输出并补齐，**逻辑门结束**与**物理播放结束**分别显示；相机读出可使用独立资源，重新触发或上传则必须等待卡完成。

In [ ]:
group_event = next(
    e for e in compiled.points[0].physical_events if e.kind == "spectrum_program"
)
rows = []
for card, item in group_event.params["cards"].items():
    program = item["program"]
    rows.append(
        {
            "card": card,
            "active_channels": item["contract"]["active_channels"],
            "segment_lengths": [s["sample_count"] for s in program["segments"]],
            "tail_samples": program["tail_samples"],
            "tail_ns": program["tail_samples"] * 0.8,
        }
    )
    assert all(s["sample_count"] % 32 == 0 for s in program["segments"])
table(rows)

### 各卡共用同一横轴

这是一次短窗口的最终 DAC 解码。横轴相对于 gate sequence 入口，而不是每卡 sample 0；触发补偿前缀已计入。
本例 B 卡未使用，所以应为零。多卡同时有门时，也按同一公共时间轴分配，不能将各卡空闲区各自删掉。
分段数量不同不会破坏逻辑对齐。软件排程一致不等于实测模拟输出同步，仍需共同参考时钟和测得的通道延迟/相位校准。

In [ ]:
from taiyidsl.spectrum_waveforms import reconstruct_window

region = next(
    e for e in compiled.points[0].logical_events if e.kind == "spectrum_region"
)
fig, ax = plt.subplots(figsize=(9, 3))
for card, item in group_event.params["cards"].items():
    contract, program = item["contract"], item["program"]
    rate = contract["sample_rate_hz"]
    sample_zero = (
        group_event.params["trigger_relative_mu"]
        * group_event.params["trigger_time_unit_s"]
        + contract["trigger_latency_samples"] / rate
    )
    n0 = max(
        0, round((region.start_mu * compiled.setup.tick_seconds - sample_zero) * rate)
    )
    n1 = min(program["physical_duration_samples"], n0 + 10_000)
    dac = reconstruct_window(program, n0, n1)
    t_us = (
        sample_zero
        + np.arange(n0, n1) / rate
        - region.start_mu * compiled.setup.tick_seconds
    ) * 1e6
    ax.plot(t_us, dac[:, 0] / 32767, label=f"{card}: output 0")
ax.set(
    xlabel="Time from gate sequence start [us]",
    ylabel="Normalized DAC",
    title="Compiled samples — not measured",
)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / "ramsey_dac.png", dpi=130)
display(fig)
# plt.close(fig)

## 8. 接实验室硬件：需要修改的只有有物理含义的配置

| 内容 | 修改位置 | 实验室需要提供什么 |
|---|---|---|
| ARTIQ 地址和实际 TTL/DDS/SPI 拓扑 | device_db | 当前设备库，与 gateware 一致 |
| 卡身份、通道、时钟、量程 | spec_db | 真实序列号、地址及硬件设置；不能继续用 SIM_A/B |
| 接线与同步组 | descriptor | 实际 TTL→卡以及逻辑信号→输出线路 |
| 量子门校准 | Qubit parameters | π/π2 幅度、时间、驱动频率、波形、frame |
| 输出时间与相位校准 | sync group 的 calibration | 测得的延迟、增益、相位及适用频率/硬件版本 |
| 准备与相机动作 | native 声明及原 ARTIQ 包 | 真实固定尾段、每 shot 清理、图像读取方式 |

修改后先重新 compile，再检查报告。`emulation=False` 不会使 compile 访问仪器；必须显式 connect/run。
下面只展示实机写法，不在 Run All 中执行：

```python
# 先按上文声明真实接线、校准快照与原生入口，再用它们重新构建实验。
lab = LabConfig(setup=real_setup, qubits=[calibrated_q0], emulation=False)
exp = ramsey_experiment(
    qubit=calibrated_q0, native=real_native, delays=delays,
    detuning=1 * kHz, second_phase=90 * deg,
    shots=100, preparation_to_gate=10 * us, group="qubit_awg",
)
compiled = lab.compile(exp)
with lab.connect(clock_session_ids=measured_clock_session_ids) as session:
    receipt = session.run(compiled)
```

clock session 标识由实验室管理，只有参考时钟连续保持、未重锁/改线时才可沿用；不能从校准文件复制一个标识来假装有效。
connect 核对身份与时钟，run 才上传、初始化和执行。缺少真实身份、校准或工具链时明确拒绝，不能静默回退 emulation。
返回的 receipt 是执行记录，不是相机图像。当前接入的读出 kernel 只负责触发；真正的探测数据须接入实验室数据读取适配器。

## 9. 保存为自己的实验包

当上面的配置已经看懂，才把它们保存到独立文件，实验正文保持短小：

```text
my_lab/
    device_db.py     # ARTIQ 设备事实
    spec_db.py       # Spectrum 卡配置
    wiring.py        # 本 notebook 完整展示的 descriptor
    qubits.py        # Qubit 校准快照
    native.py        # 原生类只登记一次
ramsey.py            # 实验参数与扫描
```

仓库中的可运行版本为 `examples/artiq_spectrum/simple_ramsey.py`，对应声明在 `ramsey_lab/`。
在项目根运行 `uv run --locked python -m examples.artiq_spectrum.simple_ramsey`，只编译和导出报告。

需要大实验时再看自动分块和 compile_stream：前者决定 AWG 分批驻留，后者控制电脑编译工作集；两者不是导入原生包的前置条件。
旧的大型 [综合教程](hybrid_from_scratch.ipynb) 保留为进阶资料。

软件验证、真实 ARTIQ 离线编译、仪器实测是三种不同证据。本 notebook 只展示前者，且不模拟原子动力学或生成假测量数据。